In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import scanpy as sc
import numpy as np


In [11]:
adata = sc.read_10x_h5('data/Xenium/breast_cancer/outs/cell_feature_matrix.h5')
adata.write_h5ad('data/Xenium/breast_cancer/outs/cell_feature_matrix.h5ad')


In [2]:
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np


In [3]:

# Step 1: Read CSV file and load all AnnData objects
csv_file_path = 'train.csv'
adata_paths = pd.read_csv(csv_file_path).iloc[:, 0]


In [4]:

# Load all AnnData objects and find the union of all genes
adatas = [sc.read_h5ad(path) for path in adata_paths]
all_genes = set(adatas[0].var_names)
for adata in adatas[1:]:
    all_genes.update(adata.var_names)

# Sort all genes to ensure consistent ordering
all_genes = sorted(all_genes)

# Function to align adata to the union of all genes
def align_adata(adata, all_genes):
    gene_to_index = {gene: i for i, gene in enumerate(adata.var_names)}
    filled_data = np.zeros((adata.shape[0], len(all_genes)))
    for i, gene in enumerate(all_genes):
        if gene in gene_to_index:
            filled_data[:, i] = adata[:, gene].X.toarray().flatten()
    return filled_data


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/VisiumHD/HumanPancreas/T_cell.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:

# Align all adatas
aligned_adatas = [align_adata(adata, all_genes) for adata in adatas]


In [ ]:

# Step 2: Define the dataset class
class AnnDataDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx]).float()


In [ ]:

# Step 3: Define the Sparse Autoencoder model
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, sparsity_penalty=1e-5):
        super(SparseAutoencoder, self).__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)
        self.sparsity_penalty = sparsity_penalty

    def forward(self, x):
        encoded = torch.relu(self.encoder(x))
        decoded = torch.relu(self.decoder(encoded))
        return encoded, decoded

    def sparsity_loss(self, encoded):
        sparsity_loss = self.sparsity_penalty * torch.mean(torch.abs(encoded))
        return sparsity_loss


In [ ]:

# Use the aligned data to determine input_dim
input_dim = len(all_genes)
hidden_dim = 64  # Adjust the hidden dimension as needed


In [ ]:

# Create a combined dataset for training
datasets = [AnnDataDataset(data) for data in aligned_adatas]
combined_dataset = torch.utils.data.ConcatDataset(datasets)
dataloader = DataLoader(combined_dataset, batch_size=32, shuffle=True)


In [ ]:

# Initialize model, loss function, and optimizer
model = SparseAutoencoder(input_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    total_loss = 0
    for data in dataloader:
        optimizer.zero_grad()
        encoded, decoded = model(data)
        loss = criterion(decoded, data) + model.sparsity_loss(encoded)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}')

# Save the trained model
torch.save(model.state_dict(), 'sparse_autoencoder.pth')
print("Model has been saved to 'sparse_autoencoder.pth'")

# Save all genes for later use
with open('all_genes.txt', 'w') as f:
    for gene in all_genes:
        f.write(f"{gene}\n")
print("All genes have been saved to 'all_genes.txt'")
